In [0]:
from pyspark.sql import functions as F

ORIGEM = "workspace.voos_bronze.posicoes_raw"
TABELA = "workspace.voos_silver.posicoes"
CHECKPOINT = "/Volumes/workspace/voos/checkpoints/silver_posicoes"

bronze = spark.readStream.table(ORIGEM)        # le so as linhas novas da bronze

s = F.col("s")
silver = (
    bronze
    .select("time", "_source_file", F.explode("states").alias("s"))
    .select(
        F.timestamp_seconds("time").alias("coleta_utc"),
        s[0].alias("icao24"),
        F.trim(s[1]).alias("voo"),
        s[2].alias("pais_origem"),
        F.timestamp_seconds(s[4].cast("long")).alias("ultimo_contato_utc"),
        s[5].cast("double").alias("longitude"),
        s[6].cast("double").alias("latitude"),
        s[7].cast("double").alias("altitude_m"),
        s[8].cast("boolean").alias("em_solo"),
        F.round(s[9].cast("double") * 3.6, 1).alias("velocidade_kmh"),
        s[10].cast("double").alias("direcao_graus"),
        s[11].cast("double").alias("taxa_vertical_ms"),
        s[14].alias("squawk"),
        "_source_file",
    )
    .where("latitude IS NOT NULL AND longitude IS NOT NULL")
    .withColumn("coleta_brt", F.from_utc_timestamp("coleta_utc", "America/Sao_Paulo"))
    .withColumn("data_coleta", F.to_date("coleta_brt"))
    .withColumn("voo", F.when(F.col("voo") == "", None).otherwise(F.col("voo")))
)

(
    silver.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABELA)
    .awaitTermination()
)
print("Silver atualizada")

Silver atualizada


In [0]:
display(spark.sql(f"""
    SELECT coleta_brt, COUNT(*) AS avioes,
           SUM(CASE WHEN em_solo THEN 1 ELSE 0 END) AS em_solo,
           ROUND(AVG(velocidade_kmh), 0) AS vel_media_kmh
    FROM {TABELA}
    GROUP BY coleta_brt
    ORDER BY coleta_brt
"""))

display(spark.table(TABELA).limit(10))

coleta_brt,avioes,em_solo,vel_media_kmh
2026-10-16T19:27:37.000Z,176,4,668.0
2026-10-16T19:29:36.000Z,181,3,671.0
2026-10-16T19:31:42.000Z,178,5,648.0
2026-10-16T19:33:42.000Z,173,6,654.0
2026-10-16T19:34:29.000Z,178,7,648.0
2026-10-16T19:36:29.000Z,182,8,644.0
2026-10-16T19:38:34.000Z,189,7,646.0
2026-10-16T19:58:24.000Z,166,8,641.0
2026-10-16T20:00:24.000Z,165,8,641.0


coleta_utc,icao24,voo,pais_origem,ultimo_contato_utc,longitude,latitude,altitude_m,em_solo,velocidade_kmh,direcao_graus,taxa_vertical_ms,squawk,_source_file,coleta_brt,data_coleta
2026-10-16T22:58:24.000Z,e80273,LPE2415,Chile,2026-10-16T22:58:21.000Z,-56.983,-20.1295,10370.82,false,758.7,292.68,0.0,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e80200,LAN708,Chile,2026-10-16T22:56:46.000Z,-55.0411,-16.1165,11277.6,false,946.2,16.84,0.0,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e07585,ARG1272,Argentina,2026-10-16T22:58:24.000Z,-45.8147,-24.5965,11277.6,false,945.1,64.34,0.65,6730,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e07584,ARG1364,Argentina,2026-10-16T22:56:21.000Z,-63.6489,-28.2648,10972.8,false,747.0,317.21,-0.33,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e07583,ARG1789,Argentina,2026-10-16T22:56:31.000Z,-58.049,-31.4399,11582.4,false,746.5,196.73,0.0,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e07581,ARG1500,Argentina,2026-10-16T22:58:23.000Z,-59.5853,-33.2409,10972.8,false,859.2,331.83,2.93,1704,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e49eed,GLO2171,Brazil,2026-10-16T22:58:23.000Z,-45.1828,-20.1665,11887.2,false,969.0,140.9,0.0,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e49ef1,GLO1238,Brazil,2026-10-16T22:58:24.000Z,-49.3094,-27.0604,10972.8,false,837.2,208.83,0.0,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e49ed5,TAM3140,Brazil,2026-10-16T22:58:23.000Z,-48.725,-26.7641,1242.06,false,385.9,248.61,0.0,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
2026-10-16T22:58:24.000Z,e48baa,TAM8075,Brazil,2026-10-16T22:54:10.000Z,-46.4433,-23.427,853.44,false,287.2,253.9,-3.25,null,/Volumes/workspace/voos/raw/2026-10-16/posicao_225825.json,2026-10-16T19:58:24.000Z,2026-10-16
